# rn222_pp_corr_s1s2 — Per-Partition Correction Pipeline (S1+S2)

Apply all 3 position-dependent corrections **per partition** for S1+S2 data.
Each correction step: tune band parameters per partition → fit → apply.

### Output
- `sr3_s1s2_corrected_for_fit_pp.pkl` — per-partition corrected events
- Comparison with globally-corrected `sr3_s1s2_corrected_for_fit.pkl`


In [ ]:
import sys
sys.path.insert(0, '/home/jietang/xenon_plot_style')
from xenonnt_plot_style import XENONPlotStyle as xps
xps.use('xenonnt')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.optimize import curve_fit
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

def safe_tight(fig=None):
    try: (fig or plt).tight_layout()
    except Exception: pass

def exp_bump_model(x, a, b, c, d, e):
    return a * np.exp((x + b) / c) * (1 - np.exp((x + b) / d)) + e

def polynomial(x, *coeffs):
    return sum(c * x**i for i, c in enumerate(coeffs))

def linear_func(x, a, b):
    return a * x + b

print('Setup complete.')


In [ ]:
def to_ts(date_str):
    return pd.Timestamp(date_str, tz='UTC').timestamp()

# 4-partition scheme (Q5-driven, Jul 27)
partitions = [
    ('P1: OLD Q5',                 to_ts('2026-05-01'), to_ts('2026-05-11')),
    ('P2: ST707, Q5 removed',       to_ts('2026-05-11'), to_ts('2026-05-26')),
    ('P3: ST707, Q5 Rn cleaned',    to_ts('2026-05-26'), to_ts('2026-06-12')),
    ('P4: FRESH Q5',                to_ts('2026-06-12'), to_ts('2026-07-02')),
]

AFT_FIT_MIN, AFT_FIT_MAX = 0.12, 0.55
AFT_REF = 0.30

for p in partitions:
    print('  %s' % p[0])

def assign_partition(start_s):
    for name, t0, t1 in partitions:
        if t0 <= start_s < t1: return name
    return None


In [ ]:
df = pd.read_pickle('/project2/lgrandi/jietang/sr3_commissioning/sr3_rn222_s1s2_pre_corr.pkl')
df['partition'] = df['time'].apply(lambda t: assign_partition(t / 1e9))
print('S1+S2 pre-correction: %d events' % len(df))
print(df['partition'].value_counts().sort_index())
print('\nColumns:', list(df.columns))


---
## Correction 1: AFT — `corr1_aft()`

Two-step centerline fit using `exp_bump_model`, then correct area to `AFT_REF = 0.30`.
Returns fitted parameters and band statistics.


In [ ]:
def corr1_aft(df, pname, upper, lower,
              aft_min=0.12, aft_max=0.55, aft_ref=0.30, n_sigma=2.0, n_bins=50):
    """
    Per-partition AFT correction (Correction 1).
    upper/lower = [a, b, c, d, e] for exp_bump_model band boundaries.
    """
    d = df[df['partition'] == pname]
    d_aft = d[(d['area_fraction_top'] >= aft_min) & (d['area_fraction_top'] <= aft_max)].copy()

    # Step 1: manual band + initial fit
    d_aft['in_band'] = (
        (d_aft['area'] < exp_bump_model(d_aft['area_fraction_top'], *upper)) &
        (d_aft['area'] > exp_bump_model(d_aft['area_fraction_top'], *lower)))

    edges = np.linspace(aft_min, aft_max, n_bins + 1)
    d_aft['aft_bin'] = pd.cut(d_aft['area_fraction_top'], bins=edges)
    meds = d_aft[d_aft['in_band']].groupby('aft_bin', observed=True)['area'].median()
    aft_c = meds.index.map(lambda iv: iv.mid).astype(float).values
    m_vals = meds.values
    good = ~np.isnan(m_vals)

    p0 = [3.70e4, -8.97e-2, -0.1, -0.05, 35e3]
    p_init, _ = curve_fit(exp_bump_model, aft_c[good], m_vals[good], p0=p0,
                          bounds=([0, -np.inf, -np.inf, -np.inf, 0], [np.inf, 0, 0, 0, np.inf]), maxfev=10000)

    # Step 2: refine with vertical cut in corrected space
    aft_arr, area_arr = d_aft['area_fraction_top'].values, d_aft['area'].values
    denom = np.maximum(exp_bump_model(aft_arr, *p_init), 1e3)
    area_corr_temp = area_arr * exp_bump_model(aft_ref, *p_init) / denom
    mask_band = d_aft['in_band'].values
    mu_rn = np.mean(area_corr_temp[mask_band])
    sig_rn = np.std(area_corr_temp[mask_band])

    d_aft['in_band_refined'] = abs(area_corr_temp - mu_rn) < sig_rn
    meds_ref = d_aft[d_aft['in_band_refined']].groupby('aft_bin', observed=True)['area'].median()
    aft_c2 = meds_ref.index.map(lambda iv: iv.mid).astype(float).values
    m_vals2 = meds_ref.values
    good2 = ~np.isnan(m_vals2)
    try:
        p_opt, _ = curve_fit(exp_bump_model, aft_c2[good2], m_vals2[good2], p0=p_init,
                              bounds=([0, -np.inf, -np.inf, -np.inf, 0], [np.inf, 0, 0, 0, np.inf]), maxfev=10000)
    except Exception:
        p_opt = p_init

    # Apply correction to FULL dataframe for this partition
    mask = (df['partition'] == pname).values
    aft_all = df.loc[mask, 'area_fraction_top'].values
    area_all = df.loc[mask, 'area'].values
    denom_all = np.maximum(exp_bump_model(aft_all, *p_opt), 1e3)
    df.loc[mask, 'area_aft_corrected'] = area_all * exp_bump_model(aft_ref, *p_opt) / denom_all

    print('  %s:' % pname.split(':')[0])
    print('    Manual band events: %d  |  Refined band events: %d  |  Total partition events: %d'
          % (mask_band.sum(), d_aft['in_band_refined'].sum(), mask.sum()))
    print('    mu_rn=%.0f PE, sig_rn=%.0f PE, +-%.0fsigma band=[%.0f, %.0f] PE'
          % (mu_rn, sig_rn, n_sigma, mu_rn - n_sigma*sig_rn, mu_rn + n_sigma*sig_rn))
    print('    Refined fit: a=%.0f e=%.0f  ref_area(AFT=%.2f)=%.0f PE'
          % (p_opt[0], p_opt[4], aft_ref, exp_bump_model(aft_ref, *p_opt)))

    # Plot: left=raw+band+CL, right=corrected+vertical band
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    aft_fine = np.linspace(0.10, 0.60, 500)

    h = ax1.hist2d(d_aft['area'], d_aft['area_fraction_top'],
                   bins=[300, 300], range=[[20e3, 100e3], [0.1, 0.6]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    plt.colorbar(h[3], ax=ax1, label='Counts')
    ax1.plot(exp_bump_model(aft_fine, *upper), aft_fine, '--', color='cyan', lw=1.5, alpha=0.6, label='Band bounds')
    ax1.plot(exp_bump_model(aft_fine, *lower), aft_fine, '--', color='cyan', lw=1.5, alpha=0.6)
    ax1.plot(exp_bump_model(aft_fine, *p_opt), aft_fine, '-', color='yellow', lw=2.5, label='CL (refined)')
    ax1.plot(m_vals2, aft_c2, 'o', color='lime', markersize=5, label='Medians refined', zorder=5)
    ax1.set_xlabel('Area [PE]'); ax1.set_ylabel('Area Fraction Top')
    ax1.set_title('%s: Raw Area vs AFT' % pname.split(':')[0])
    ax1.legend(fontsize=8, loc='lower right'); ax1.grid(True, alpha=0.2)

    d_plot = df[(df['partition'] == pname) & (df['area_fraction_top'] >= aft_min) & (df['area_fraction_top'] <= aft_max)]

    # Select vertical band events (corrected area within +-n_sigma band)
    d_band = d_plot[abs(d_plot['area_aft_corrected'] - mu_rn) < n_sigma * sig_rn]
    pp_mu = d_band['area_aft_corrected'].median()

    h = ax2.hist2d(d_plot['area_aft_corrected'], d_plot['area_fraction_top'],
                   bins=[300, 300], range=[[20e3, 100e3], [0.1, 0.6]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    plt.colorbar(h[3], ax=ax2, label='Counts')
    ax2.axvline(x=pp_mu, color='yellow', lw=2, alpha=0.7,
                label='P median (band) = %.0f PE' % pp_mu)
    ax2.axvline(x=mu_rn - n_sigma*sig_rn, color='cyan', ls='--', lw=1.5,
                label='+-%.0fsigma = [%.0f, %.0f] PE' % (n_sigma, mu_rn-n_sigma*sig_rn, mu_rn+n_sigma*sig_rn))
    ax2.axvline(x=mu_rn + n_sigma*sig_rn, color='cyan', ls='--', lw=1.5)
    # Per-AFT-bin median of CORRECTED area within vertical band
    afb = np.linspace(aft_min, aft_max, 16)
    d_band_bin = d_band.copy()
    d_band_bin['ab'] = pd.cut(d_band_bin['area_fraction_top'], bins=afb)
    meds_b = d_band_bin.groupby('ab', observed=True)['area_aft_corrected'].median()
    mids = meds_b.index.map(lambda iv: iv.mid).astype(float).values
    ax2.plot(meds_b.values, mids, 'o-', color='darkorange', lw=1, ms=3,
             label='Median per AFT bin (band)')
    ax2.set_xlabel('AFT-Corrected Area [PE]'); ax2.set_ylabel('Area Fraction Top')
    ax2.set_title('%s: Corrected (should be vertical)' % pname.split(':')[0])
    ax2.legend(fontsize=8, loc='lower right', framealpha=0.85); ax2.grid(True, alpha=0.2)
    safe_tight(fig); plt.show()

    return p_opt, mu_rn, sig_rn


---
## Correction 2: r² vs AFT — `corr2_r2_aft()`

Select Rn-band with vertical cut in `area_aft_corrected`, fit polynomial r²(AFT), correct r².


In [ ]:
def corr2_r2_aft(df, pname, area_lo, area_hi,
                 aft_min=0.12, aft_max=0.55, aft_ref=0.30, poly_deg=10,
                 r2_min=0, r2_max=750, n_bins=40):
    """
    Per-partition r² vs AFT correction (Correction 2).
    area_lo/hi: vertical band in area_aft_corrected to select Rn events.
    """
    mask_p = df['partition'] == pname
    mask_rn = (df['area_aft_corrected'] >= area_lo) & (df['area_aft_corrected'] <= area_hi)
    mask_fit = (df['area_fraction_top'] >= aft_min) & (df['area_fraction_top'] <= aft_max) & (df['r2'] >= r2_min) & (df['r2'] <= r2_max)
    d_fit = df[mask_p & mask_rn & mask_fit].copy()

    edges = np.linspace(aft_min, aft_max, n_bins + 1)
    d_fit['aft_bin'] = pd.cut(d_fit['area_fraction_top'], bins=edges)
    meds = d_fit.groupby('aft_bin', observed=True)['r2'].median()
    cnts = d_fit.groupby('aft_bin', observed=True)['r2'].count()
    aft_c = meds.index.map(lambda iv: iv.mid).astype(float).values
    r2_m = meds.values
    good = (cnts.values >= 20)

    p0 = [np.median(r2_m[good])] + [0.0] * poly_deg
    popt, _ = curve_fit(polynomial, aft_c[good], r2_m[good], p0=p0, maxfev=50000)
    r2_ref = polynomial(aft_ref, *popt)

    # Apply correction to FULL partition
    m = df['partition'] == pname
    cf = r2_ref / np.maximum(polynomial(df.loc[m, 'area_fraction_top'].values, *popt), 0.1)
    df.loc[m, 'r2_aft_corrected'] = df.loc[m, 'r2'] * cf

    print('  %s: Rn-band events=%d  r2_ref(%.2f)=%.0f  cf=[%.3f, %.3f]'
          % (pname.split(':')[0], len(d_fit), aft_ref, r2_ref, cf.min(), cf.max()))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    h = ax1.hist2d(d_fit['area_fraction_top'], d_fit['r2'],
                   bins=[100, 100], range=[[aft_min, aft_max], [r2_min, r2_max]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    plt.colorbar(h[3], ax=ax1, label='Counts')
    aft_fine = np.linspace(aft_min, aft_max, 200)
    ax1.plot(aft_fine, polynomial(aft_fine, *popt), 'r-', lw=2, label='Poly deg %d' % poly_deg)
    ax1.set_xlabel('Area Fraction Top'); ax1.set_ylabel('r2 [cm2]')
    ax1.set_title('%s: AFT vs r2 (Rn-band)' % pname.split(':')[0])
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.2)

    ax2.errorbar(aft_c[good], r2_m[good],
                 yerr=np.sqrt(np.maximum(r2_m[good], 1))/np.sqrt(cnts.values[good]),
                 fmt='o', capsize=3, color='steelblue', label='Median r2 bin')
    ax2.plot(aft_fine, polynomial(aft_fine, *popt), 'r-', lw=2, label='Poly fit')
    ax2.set_xlabel('Area Fraction Top'); ax2.set_ylabel('Median r2 [cm2]')
    ax2.set_title('%s: AFT vs median r2 + fit' % pname.split(':')[0])
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
    safe_tight(fig); plt.show()
    return popt


---
## Correction 3: r² vs Area — `corr3_area_r2()`

Select band with linear boundaries in r2_aft_corrected vs area_aft_corrected space,
fit polynomial, correct area.


In [ ]:
def corr3_area_r2(df, pname, k_slope, intercept_upper, intercept_lower,
                  r2_min=0, r2_max=100, r2_fit_max=55, poly_deg=6, n_bins=30):
    """
    Per-partition r2 vs Area correction (Correction 3).
    Band: area < k_slope*r2 + intercept_upper  AND  area > k_slope*r2 + intercept_lower.
    """
    mask_p = df['partition'] == pname
    mask_fit = (df['r2_aft_corrected'] >= r2_min) & (df['r2_aft_corrected'] <= r2_max) & (df['area_aft_corrected'] >= 0) & (df['area_aft_corrected'] <= 1e5)
    d_fit = df[mask_p & mask_fit].copy()

    d_fit['in_r2_band'] = (
        (d_fit['area_aft_corrected'] < k_slope*d_fit['r2_aft_corrected'] + intercept_upper) &
        (d_fit['area_aft_corrected'] > k_slope*d_fit['r2_aft_corrected'] + intercept_lower))

    edges = np.linspace(r2_min, r2_max, n_bins + 1)
    d_fit['r2_bin'] = pd.cut(d_fit['r2_aft_corrected'], bins=edges)
    meds = d_fit[d_fit['in_r2_band']].groupby('r2_bin', observed=True)['area_aft_corrected'].median()
    cnts = d_fit[d_fit['in_r2_band']].groupby('r2_bin', observed=True)['area_aft_corrected'].count()
    r2_c = meds.index.map(lambda iv: iv.mid).astype(float).values
    a_m = meds.values
    good = (cnts.values >= 10) & (r2_c < r2_fit_max)

    p0 = [np.median(a_m[good])] + [0.0] * poly_deg
    popt, _ = curve_fit(polynomial, r2_c[good], a_m[good], p0=p0, maxfev=50000)
    area_ref = polynomial(0, *popt)

    # Apply correction to FULL partition
    m = df['partition'] == pname
    cf_raw = area_ref / polynomial(df.loc[m, 'r2_aft_corrected'].values, *popt)
    cf = np.clip(cf_raw, 0.5, 2.0)
    df.loc[m, 'area_2_corrected'] = df.loc[m, 'area_aft_corrected'] * cf
    n_clip = (cf_raw != cf).sum()

    print('  %s: band events=%d  area_ref(r2=0)=%.0f PE  cf=[%.3f,%.3f]%s'
          % (pname.split(':')[0], d_fit['in_r2_band'].sum(), area_ref, cf.min(), cf.max(),
             ' (clipped %d)' % n_clip if n_clip > 0 else ''))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    h = ax1.hist2d(d_fit['r2_aft_corrected'], d_fit['area_aft_corrected'],
                   bins=[80, 100], range=[[r2_min, r2_max], [30e3, 100e3]],
                   cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
    plt.colorbar(h[3], ax=ax1, label='Counts')
    r2f = np.linspace(r2_min, r2_max, 200)
    ax1.plot(r2f, k_slope*r2f + intercept_upper, '--', color='cyan', lw=2, alpha=0.7, label='Band bound')
    ax1.plot(r2f, k_slope*r2f + intercept_lower, '--', color='cyan', lw=2, alpha=0.7)
    ax1.plot(r2f, polynomial(r2f, *popt), 'r-', lw=2, label='Poly deg %d' % poly_deg)
    ax1.axvline(x=r2_fit_max, color='orange', ls=':', lw=1.5, label='Fit limit r2=%d' % r2_fit_max)
    ax1.set_xlabel('r2_aft_corrected [cm2]'); ax1.set_ylabel('area_aft_corrected [PE]')
    ax1.set_title('%s: r2 vs area + band' % pname.split(':')[0])
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.2)

    ax2.errorbar(r2_c[good], a_m[good],
                 yerr=np.sqrt(np.maximum(a_m[good],1))/np.sqrt(cnts.values[good]),
                 fmt='o', capsize=3, color='steelblue', label='Median area bin')
    ax2.plot(r2f, polynomial(r2f, *popt), 'r-', lw=2, label='Poly fit')
    ax2.axhline(area_ref, color='gray', ls='--', lw=1.5, label='Ref(0)=%.0f' % area_ref)
    ax2.axvline(x=r2_fit_max, color='orange', ls=':', lw=1.5)
    ax2.set_xlabel('r2_aft_corrected [cm2]'); ax2.set_ylabel('Median area [PE]')
    ax2.set_title('%s: r2 vs median area + fit' % pname.split(':')[0])
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
    safe_tight(fig); plt.show()
    return popt, area_ref


---
## Correction 1: AFT — Per-Partition

Tune `upper`/`lower` per partition to enclose the Rn-222/220 alpha band.
These are `exp_bump_model` parameters `[a, b, c, d, e]`.
The function outputs diagnostic prints and a 2-panel before/after figure.


In [ ]:
# ── P1 ──
corr1_aft(df, 'P1: OLD Q5',
          upper=[8.86e+04, -0.0600, -0.0900, -0.0600, 68000],   # ← tune
          lower=[7.30e+04, -0.0600, -0.0900, -0.0600, 58000],   # ← tune
          aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF, n_sigma=2.0)


In [ ]:
# ── P2 ──
corr1_aft(df, 'P2: ST707, Q5 removed',
          upper=[8.86e+04, -0.0600, -0.0900, -0.0600, 68000],   # ← tune
          lower=[7.30e+04, -0.0600, -0.0900, -0.0600, 58000],   # ← tune
          aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF, n_sigma=2.0)


In [ ]:
# ── P3 ──
corr1_aft(df, 'P3: ST707, Q5 Rn cleaned',
          upper=[8.86e+04, -0.0600, -0.0900, -0.0600, 68000],   # ← tune
          lower=[7.30e+04, -0.0600, -0.0900, -0.0600, 58000],   # ← tune
          aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF, n_sigma=2.0)


In [ ]:
# ── P4 ──
corr1_aft(df, 'P4: FRESH Q5',
          upper=[8.86e+04, -0.0600, -0.0900, -0.0600, 68000],   # ← tune
          lower=[7.30e+04, -0.0600, -0.0900, -0.0600, 58000],   # ← tune
          aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF, n_sigma=2.0)


---
## Correction 2: r² vs AFT — Per-Partition

Tune `area_lo`/`area_hi` per partition. These define a vertical band in
`area_aft_corrected` (from Correction 1) to select Rn-222 events.
The band should be centered on the corrected Rn peak.


In [ ]:
# ── P1 ──
# Check the corrected peak from Corr 1 output, set area_lo/hi around it
corr2_r2_aft(df, 'P1: OLD Q5', area_lo=40000, area_hi=45000,  # ← tune
             aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF)


In [ ]:
# ── P2 ──
# Check the corrected peak from Corr 1 output, set area_lo/hi around it
corr2_r2_aft(df, 'P2: ST707, Q5 removed', area_lo=40000, area_hi=45000,  # ← tune
             aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF)


In [ ]:
# ── P3 ──
# Check the corrected peak from Corr 1 output, set area_lo/hi around it
corr2_r2_aft(df, 'P3: ST707, Q5 Rn cleaned', area_lo=40000, area_hi=45000,  # ← tune
             aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF)


In [ ]:
# ── P4 ──
# Check the corrected peak from Corr 1 output, set area_lo/hi around it
corr2_r2_aft(df, 'P4: FRESH Q5', area_lo=40000, area_hi=45000,  # ← tune
             aft_min=AFT_FIT_MIN, aft_max=AFT_FIT_MAX, aft_ref=AFT_REF)


---
## Correction 3: r² vs Area — Per-Partition

Tune `k_slope`, `intercept_upper`, `intercept_lower` per partition.
Band: `area < k_slope*r2 + intercept_upper` AND `area > k_slope*r2 + intercept_lower`.
The Rn band slopes downward with increasing r².


In [ ]:
# ── P1 ──
corr3_area_r2(df, 'P1: OLD Q5',
              k_slope=-110, intercept_upper=75000, intercept_lower=68000,  # ← tune
              r2_min=0, r2_max=100, r2_fit_max=55, poly_deg=6)


In [ ]:
# ── P2 ──
corr3_area_r2(df, 'P2: ST707, Q5 removed',
              k_slope=-110, intercept_upper=75000, intercept_lower=68000,  # ← tune
              r2_min=0, r2_max=100, r2_fit_max=55, poly_deg=6)


In [ ]:
# ── P3 ──
corr3_area_r2(df, 'P3: ST707, Q5 Rn cleaned',
              k_slope=-110, intercept_upper=75000, intercept_lower=68000,  # ← tune
              r2_min=0, r2_max=100, r2_fit_max=55, poly_deg=6)


In [ ]:
# ── P4 ──
corr3_area_r2(df, 'P4: FRESH Q5',
              k_slope=-110, intercept_upper=75000, intercept_lower=68000,  # ← tune
              r2_min=0, r2_max=100, r2_fit_max=55, poly_deg=6)


---
## Apply Fiducial Cuts

Same cuts as original analysis. Applied after all 3 corrections.
- AFT cut: `[0.1, 0.55]`
- FV cut: `AFT ∈ [0.15, 0.455]`, `r2_aft_corrected < 3800`


In [ ]:
# AFT fiducial cut
AFT_MIN, AFT_MAX = 0.10, 0.55
m_aft = (df['area_fraction_top'] > AFT_MIN) & (df['area_fraction_top'] < AFT_MAX)
print('AFT cut [%.2f, %.2f]:' % (AFT_MIN, AFT_MAX))
print('  Before: %d  |  After: %d  |  Removed: %d (%.1f%%)'
      % (len(df), m_aft.sum(), (~m_aft).sum(), 100*(~m_aft).sum()/len(df)))

# FV cut
AFT_FV_MIN, AFT_FV_MAX = 0.15, 0.455
R2_FV_MAX = 75
m_fv = (m_aft &
        (df['area_fraction_top'] >= AFT_FV_MIN) & (df['area_fraction_top'] <= AFT_FV_MAX) &
        (df['r2_aft_corrected'] >= 0) & (df['r2_aft_corrected'] <= R2_FV_MAX))
print('FV cut [AFT %.2f-%.2f, r2<%.0f (S1+S2)]:' % (AFT_FV_MIN, AFT_FV_MAX, R2_FV_MAX))
print('  Before: %d  |  After: %d  |  Removed: %d (%.1f%%)'
      % (m_aft.sum(), m_fv.sum(), (m_aft & ~m_fv).sum(), 100*(m_aft & ~m_fv).sum()/m_aft.sum()))

df_cut = df[m_fv].copy().reset_index(drop=True)
print('\nFinal events after all cuts: %d' % len(df_cut))
print('Per partition:')
print(df_cut['partition'].value_counts().sort_index())


In [ ]:
# ═════════════════════════════════════════════════════════════
# CORRECTION PIPELINE COMPARISON — all stages (after cuts)
# ═════════════════════════════════════════════════════════════

stages = [
    ('Raw',                    'area'),
    ('AFT-Corrected',          'area_aft_corrected'),
    ('AFT + r2 Corrected',     'area_2_corrected'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, (label, col) in zip(axes, stages):
    if col in df_cut.columns:
        h = ax.hist2d(df_cut[col], df_cut['area_fraction_top'],
                      bins=[300, 200],
                      range=[[20e3, 100e3], [0.1, 0.65]],
                      cmin=1, norm=matplotlib.colors.LogNorm(), cmap='viridis')
        plt.colorbar(h[3], ax=ax, label='Counts')
        ax.set_xlabel('Area [PE]')
        ax.set_ylabel('Area Fraction Top')
        ax.set_title(label, fontweight='bold', fontsize=12)
        ax.grid(True, alpha=0.2)

fig.suptitle('Area vs AFT — Correction Stages (after AFT+FV cuts)', fontsize=14, fontweight='bold', y=1.02)
safe_tight(fig)
plt.show()


In [ ]:
save_path = '/project2/lgrandi/jietang/sr3_commissioning/sr3_s1s2_corrected_for_fit_pp.pkl'
df_cut.to_pickle(save_path)
print('Saved: %s (%d events, %d columns)' % (save_path, len(df_cut), len(df_cut.columns)))


---
## Compare: Per-Partition vs Global Correction

Load the original globally-corrected pickle and compare peak positions
and band flatness per partition.


In [ ]:
df_global = pd.read_pickle('/project2/lgrandi/jietang/sr3_commissioning/sr3_s1s2_corrected_for_fit.pkl')
df_global['partition'] = df_global['time'].apply(lambda t: assign_partition(t / 1e9))
print('Global-corrected: %d events' % len(df_global))
print(df_global['partition'].value_counts().sort_index())

# 1D corrected area spectra overlay per partition
n_p = len(partitions)
fig, axes = plt.subplots(n_p, 1, figsize=(10, 3 * n_p), sharex=True)
if n_p == 1: axes = [axes]
for ax, (pname, _, _) in zip(axes, partitions):
    d_pp = df_cut[df_cut['partition'] == pname]
    d_gl = df_global[df_global['partition'] == pname]
    if len(d_pp) == 0: continue
    col_pp = 'area_2_corrected'
    col_gl = [c for c in ['area_2_corrected', 'area_fv_corrected', 'area_corrected'] if c in d_gl.columns]
    col_gl = col_gl[0] if col_gl else d_gl.columns[-1]
    ax.hist(d_gl[col_gl]/1e3, bins=200, range=[35, 65], histtype='step', color='gray', lw=2, label='Global')
    ax.hist(d_pp[col_pp]/1e3, bins=200, range=[35, 65], histtype='step', color='steelblue', lw=2, label='PP')
    ax.set_ylabel(pname.split(':')[0], fontsize=9, fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.15)
axes[-1].set_xlabel('Corrected Area [103 PE]', fontsize=12)
fig.suptitle('S1+S2: PP vs Global — 1D Corrected Area Spectra', fontsize=14, fontweight='bold')
safe_tight(fig); plt.show()


In [ ]:
# ═════════════════════════════════════════════════════════════
# Gaussian fit on Rn-222 peak — compare PP vs Global
# ═════════════════════════════════════════════════════════════

from scipy.optimize import curve_fit

def gauss(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma)**2)

# Determine fit range from data
col_pp = 'area_2_corrected'
col_gl = [c for c in df_global.columns if 'corrected' in c][0]

all_pp = df_cut[col_pp].values
all_gl = df_global[col_gl].values
fit_range = (np.median(all_pp) - 10000, np.median(all_pp) + 10000)

print('Rn-222 peak Gaussian fit comparison (PP vs Global):')
print('%-30s %14s %14s %12s %12s' % ('Partition', 'PP mu [PE]', 'Global mu [PE]', 'Delta mu', 'PP/Gl sigma'))
print('-' * 85)

mus_pp, mus_gl, sigs_pp, sigs_gl = [], [], [], []
fig, axes = plt.subplots(len(partitions), 1, figsize=(12, 3 * len(partitions)), sharex=True)
if len(partitions) == 1: axes = [axes]

for ax, (pname, _, _) in zip(axes, partitions):
    d_pp = df_cut[df_cut['partition'] == pname]
    d_gl = df_global[df_global['partition'] == pname]
    if len(d_pp) < 50: continue

    # Fit Gaussians
    pp_vals = d_pp[col_pp].values
    gl_vals = d_gl[col_gl].values

    y_pp, x_pp = np.histogram(pp_vals, bins=200, range=fit_range)
    x_pp_c = 0.5 * (x_pp[:-1] + x_pp[1:])
    y_gl, x_gl = np.histogram(gl_vals, bins=200, range=fit_range)
    x_gl_c = 0.5 * (x_gl[:-1] + x_gl[1:])

    # Initial guess for PP
    imax_pp = np.argmax(y_pp)
    p0_pp = [y_pp[imax_pp], x_pp_c[imax_pp], 800]
    try:
        popt_pp, _ = curve_fit(gauss, x_pp_c, y_pp, p0=p0_pp,
                               bounds=([0, fit_range[0], 100], [np.inf, fit_range[1], 5000]))
    except Exception:
        popt_pp = p0_pp

    # Initial guess for Global
    imax_gl = np.argmax(y_gl)
    p0_gl = [y_gl[imax_gl], x_gl_c[imax_gl], 800]
    try:
        popt_gl, _ = curve_fit(gauss, x_gl_c, y_gl, p0=p0_gl,
                               bounds=([0, fit_range[0], 100], [np.inf, fit_range[1], 5000]))
    except Exception:
        popt_gl = p0_gl

    mus_pp.append(popt_pp[1]); mus_gl.append(popt_gl[1])
    sigs_pp.append(popt_pp[2]); sigs_gl.append(popt_gl[2])

    print('%-30s %14d %14d %12d %15s' % (
        pname, int(popt_pp[1]), int(popt_gl[1]), int(popt_pp[1] - popt_gl[1]),
        '%.0f / %.0f' % (popt_pp[2], popt_gl[2])))

    # Plot
    ax.bar(x_pp_c, y_pp, width=x_pp_c[1]-x_pp_c[0], alpha=0.3, color='steelblue', label='PP')
    ax.bar(x_gl_c, y_gl, width=x_gl_c[1]-x_gl_c[0], alpha=0.3, color='gray', label='Global')
    xf = np.linspace(fit_range[0], fit_range[1], 500)
    ax.plot(xf, gauss(xf, *popt_pp), '-', color='steelblue', lw=2)
    ax.plot(xf, gauss(xf, *popt_gl), '-', color='black', lw=2)
    ax.set_ylabel(pname.split(':')[0], fontsize=9, fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.15)

axes[-1].set_xlabel('Corrected Area [PE]', fontsize=12)
fig.suptitle('Gaussian Fit on Rn-222 Peak — PP vs Global', fontsize=14, fontweight='bold')
safe_tight(fig); plt.show()

print('-' * 85)
if mus_pp:
    print('PP mu spread:     %d PE  |  PP sigma spread:     %d PE' % (
        int(max(mus_pp) - min(mus_pp)), int(max(sigs_pp) - min(sigs_pp))))
    print('Global mu spread: %d PE  |  Global sigma spread: %d PE' % (
        int(max(mus_gl) - min(mus_gl)), int(max(sigs_gl) - min(sigs_gl))))
